In [1]:
import pandas as pd

# Cleaning the data
### 1. Characters df
##### Removing columns that are not needed

In [2]:
dfCharacters = pd.read_csv("../data/raw/characters.csv")

dfCharacters = dfCharacters.drop(columns=["url",
                                          "name_kanji",
                                          "about"],
                                 errors="ignore")

### Normalizing
##### Favorites cast to int and set NaN to 0

In [3]:
dfCharacters.drop_duplicates()
dfCharacters["favorites"] = (
    pd.to_numeric(dfCharacters["favorites"], errors="coerce")
      .fillna(0)
      .astype(int)
)

##### ID set to int and removed Nan values

In [4]:
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != 0]
dfCharacters["character_mal_id"] = (
    pd.to_numeric(dfCharacters["character_mal_id"], errors="coerce")
        .fillna(-1)
        .astype(int)
)
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != -1]
dfCharacters

,character_mal_id,name,image,favorites
0,280386,Envi Mel Champagne,https://cdn.myanimelist.net/images/characters/...,0
1,280354,Eleven,https://cdn.myanimelist.net/images/characters/...,0
2,280353,Stud,https://cdn.myanimelist.net/images/characters/...,0
3,280352,Judge,https://cdn.myanimelist.net/images/characters/...,0
4,280339,Eiji Kurokawa,https://cdn.myanimelist.net/img/sp/icon/apple-...,0
...,...,...,...,...
209958,282276,Farrah Van Dorothy,https://cdn.myanimelist.net/images/characters/...,0
209959,282277,Harris Mead,https://cdn.myanimelist.net/images/characters/...,0
209960,282278,Rob,https://cdn.myanimelist.net/images/characters/...,0
209961,282281,Grimm,https://cdn.myanimelist.net/images/characters/...,0


### 2. Nicknames df
##### Removed rows where nickname was empty or Nan, useless
##### Some Characters have multiple nicknames, so it's better to put them in the same row as a list

In [5]:
dfNicknames = pd.read_csv("../data/raw/character_nicknames.csv")

dfNicknames = dfNicknames[dfNicknames["nickname"].notna()]

dfNicknames = (
    dfNicknames
    .groupby("character_mal_id")["nickname"]
    .apply(list)
    .reset_index()
)

In [6]:
dfNicknames

,character_mal_id,nickname
0,3,"[Running Rock, Black Dog]"
1,5,"[Ichi-nii, Shinigami Daiko (Substitute Soul Re..."
2,7,[Hime]
3,11,"[Ed, Fullmetal Alchemist, Hagane no shounen, C..."
4,12,"[Al, Armored Alchemist]"
...,...,...
30262,282048,[Great Zuma]
30263,282159,"[Ling Long, Silvermoon]"
30264,282227,[Mei's Mother]
30265,282254,[Cyrano]


# Merging

In [7]:
dfCharacters = dfCharacters.merge(dfNicknames, on="character_mal_id", how = "left")
dfNicknames = None

In [8]:
dfCharacters['nickname'] = dfCharacters['nickname'].apply(lambda x: x if isinstance(x, list) else [])

### 3. AnimeWorks df

In [9]:
dfAnimeWorks = pd.read_csv("../data/raw/character_anime_works.csv")
dfCharacters = dfCharacters.merge(dfAnimeWorks, on="character_mal_id")
dfAnimeWorks = None
dfCharacters = dfCharacters.drop(columns=["name"]) # Redundant column, keeping the one with the comma
dfCharacters

,character_mal_id,image,favorites,nickname,anime_mal_id,character_name,role
0,280386,https://cdn.myanimelist.net/images/characters/...,0,[],59846,"Champagne, Envi Mel",Supporting
1,280354,https://cdn.myanimelist.net/images/characters/...,0,[],60071,Eleven,Supporting
2,280353,https://cdn.myanimelist.net/images/characters/...,0,[],60071,Stud,Supporting
3,280352,https://cdn.myanimelist.net/images/characters/...,0,[],60071,Judge,Supporting
4,280339,https://cdn.myanimelist.net/img/sp/icon/apple-...,0,[],60531,"Kurokawa, Eiji",Supporting
...,...,...,...,...,...,...,...
238936,282276,https://cdn.myanimelist.net/images/characters/...,0,[],3258,"Van Dorothy, Farrah",Supporting
238937,282277,https://cdn.myanimelist.net/images/characters/...,0,[],3258,"Mead, Harris",Supporting
238938,282278,https://cdn.myanimelist.net/images/characters/...,0,[],7625,Rob,Supporting
238939,282281,https://cdn.myanimelist.net/images/characters/...,0,[],7625,Grimm,Supporting


In [10]:
dfTemp = (
    dfCharacters
    .groupby(["character_mal_id", "role"])
    .agg({
        "favorites": "max",
        "anime_mal_id": lambda x: [int(v) for v in x.unique()]
    })
    .reset_index()
)
idx = (
    dfCharacters
    .groupby("character_mal_id")["favorites"]
    .idxmax()
)
dfCharacters = dfCharacters.loc[idx][["character_mal_id", "nickname", "character_name"]]
dfTemp = dfTemp.merge(dfCharacters, on="character_mal_id")
dfTemp

,character_mal_id,role,favorites,anime_mal_id,nickname,character_name
0,1,Main,48344,"[1, 5, 4037]",[],"Spiegel, Spike"
1,2,Main,9535,"[1, 5, 4037]",[],"Valentine, Faye"
2,3,Main,2249,"[1, 5, 4037]","[Running Rock, Black Dog]","Black, Jet"
3,4,Main,2391,[17205],[],Ein
4,4,Supporting,2391,"[1, 5, 4037]",[],Ein
...,...,...,...,...,...,...
150575,282276,Supporting,0,[3258],[],"Van Dorothy, Farrah"
150576,282277,Supporting,0,[3258],[],"Mead, Harris"
150577,282278,Supporting,0,[7625],[],Rob
150578,282281,Supporting,0,[7625],[],Grimm


In [11]:
dfCharacters = None
dfTemp.to_csv("../data/cleaned/anime_characters.csv", index=False)
dfTemp = None